# Brain Age: Leave-One-Site-Out Cross Validation
Questo notebook esegue un addestramento partendo da zero e convalida il modello utilizzando una **Cross-Validation basata sui siti clinici (Guys, HH, IOP)**. 
Non c'è un Test Set isolato: le prestazioni del modello vengono valutate misurando quanto è bravo a generalizzare su un *intero ospedale/scanner* (Validation Set) dopo essersi addestrato sugli altri due ospedali.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
import os
sys.path.append(os.path.abspath('./SFCN'))

In [ ]:
import os
import pandas as pd
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from datetime import datetime
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr

from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu

MODELS_DIR = '/kaggle/working/models'
PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(MODELS_DIR, exist_ok=True)

os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Configurazione

In [ ]:
# --- PERCORSI DATI ---
CSV_PATH = "/kaggle/input/ixi-dataset/ixi_info.csv"
DATA_DIR = "/kaggle/input/ixi-dataset/Prep" # Cartella root che contiene Prep_Guys, Prep_HH, Prep_IOP

CORRECTED_DIR = "/kaggle/input/datasets/collab4444/ixi-t1-corrected-items/corrected_IXI_subjects_T1w"
MODALITY_SUFFIX = "" # Oppure "_T1" o "_T2" a seconda dei nomi delle tue cartelle

# --- CONFIGURAZIONE IMMAGINI GENERATE (TRAINING) ---
USE_GENERATED_FOR_TRAIN = True
GENERATION_MODEL = "EAGAN" # Es. "EAGAN", "CYCLEGAN", ecc. (case-insensitive)
GEN_DATA_DIR = "/kaggle/input/datasets/collab2000/risultati-dataset-2/dataset 2/T1 - T2/T1 to T2"

ALL_SITES = ['Guys', 'HH', 'IOP']

# --- IPERPARAMETRI ---
EPOCHS = 40
BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 15

# --- CALCOLO DINAMICO DEL RANGE DI ETÀ ---
import math

df_temp = pd.read_csv(CSV_PATH)
reference_date_temp = datetime(2015, 2, 23)
ages_temp = []
for idx, row in df_temp.iterrows():
    dob_str = row['DOB']
    if pd.isna(dob_str): continue
    try:
        dob = datetime.strptime(str(dob_str).split(" ")[0], "%Y-%m-%d")
        ages_temp.append((reference_date_temp - dob).days / 365.25)
    except:
        pass

MIN_AGE = math.floor(min(ages_temp))
MAX_AGE = math.ceil(max(ages_temp))
OUTPUT_DIM = MAX_AGE - MIN_AGE
BIN_RANGE = [MIN_AGE, MAX_AGE]

print(f"Calculated ages from dataset: Min {min(ages_temp):.2f}, Max {max(ages_temp):.2f}")
print(f"Bin Adaptation (SFCN paper style): Range {BIN_RANGE}, Output Dimension (N. Bins) = {OUTPUT_DIM}")


## 2. Site-Aware Dataloader
Il Dataloader carica solo i pazienti provenienti dai siti specificati nella lista `allowed_sites`.

In [ ]:
class IXISiteDataset(Dataset):
    import re
    def __init__(self, data_dir, csv_path, allowed_sites, is_train=False):
        self.samples = []
        self.is_train = is_train
        self.bin_range = BIN_RANGE
        self.bin_step = 1
        self.sigma = 1.0
        
        df = pd.read_csv(csv_path)
        reference_date = datetime(2015, 2, 23)
        
        for site in allowed_sites:
            # Se siamo in addestramento E stiamo usando le immagini generate
            if self.is_train and USE_GENERATED_FOR_TRAIN:
                # Esempio: /.../PREP_GUYS/Predizioni_EAGAN/
                folder_path = os.path.join(GEN_DATA_DIR, f"PREP_{site.upper()}", f"Predizioni_{GENERATION_MODEL.upper()}")
                if not os.path.exists(folder_path):
                    print(f"Warning: Generated images folder for site {site} not found: {folder_path}")
                    continue
            else:
                # Altrimenti usiamo le immagini REALI (Validazione o Training classico)
                folder_name = f"Prep_{site}{MODALITY_SUFFIX}"
                folder_path = os.path.join(data_dir, folder_name)
                
                if not os.path.exists(folder_path):
                    # Fallback senza suffisso
                    folder_path = os.path.join(data_dir, f"Prep_{site}")
                    if not os.path.exists(folder_path):
                        print(f"Warning: Real folder for site {site} not found.")
                        continue
                    
            for file in os.listdir(folder_path):
                if not (file.startswith("registered_image_") and file.endswith(".nii")):
                    continue
                    
                nii_path = os.path.join(folder_path, file)
                
                # Sostituzione file corrotto (solo per immagini reali)
                if not (self.is_train and USE_GENERATED_FOR_TRAIN):
                    corrected_path = os.path.join(CORRECTED_DIR, site, file)
                    if os.path.exists(corrected_path):
                        nii_path = corrected_path
                    
                # Regex flessibile che cattura l'ID indipendentemente dai suffissi extra
                import re
                match = re.search(r'registered_image_(\d+)', file)
                if not match:
                    continue
                ixi_id_str = match.group(1)
                
                try:
                    ixi_id = int(ixi_id_str)
                except ValueError:
                    continue
                    
                row = df[df['IXI_ID'] == ixi_id]
                if len(row) == 0:
                    continue
                    
                dob_str = row.iloc[0]['DOB']
                if pd.isna(dob_str):
                    continue
                    
                try:
                    dob_str = str(dob_str).split(" ")[0]
                    dob = datetime.strptime(dob_str, "%Y-%m-%d")
                    true_age = (reference_date - dob).days / 365.25
                    
                    y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                    
                    self.samples.append({
                        "ixi_id": ixi_id,
                        "site": site,
                        "nii_path": nii_path,
                        "label_vect": y,
                        "true_age": true_age
                    })
                except Exception as e:
                    continue
                    
        data_type = "GENERATED" if (is_train and USE_GENERATED_FOR_TRAIN) else "REAL"
        print(f"[{'TRAIN' if is_train else 'VAL'}] Loaded {len(self.samples)} {data_type} patients (Sites: {allowed_sites}).")

    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            dx = np.random.randint(-3, 4)
            dy = np.random.randint(-3, 4)
            dz = np.random.randint(-3, 4)
            # Data augmentation: Random horizontal flip
            if np.random.rand() > 0.5:
                data = np.flip(data, axis=0).copy()
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        

        return tensor_data, label_vect, sample['true_age'], sample['site']

## 3. Training Loop (Leave-One-Site-Out)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_site_model(train_sites, val_site):
    print(f"\n{'='*50}")
    print(f" FOLD: Testing on site [{val_site}]")
    print(f" Training on: {train_sites}")
    print(f"{'='*50}")
    
    train_dataset = IXISiteDataset(DATA_DIR, CSV_PATH, train_sites, is_train=True)
    val_dataset = IXISiteDataset(DATA_DIR, CSV_PATH, [val_site], is_train=False)
    
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print("Empty dataset. Skipping fold.")
        return None, None, None, None
        
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Inizializziamo DA ZERO (Nessun pretraining caricato)
    model = SFCN(output_dim=OUTPUT_DIM)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.KLDivLoss(reduction='batchmean')
    
    best_val_mae = float('inf')
    epochs_no_improve = 0
    best_model_path = os.path.join(MODELS_DIR, f'best_model_val_{val_site}.pth')
    
    bin_centers = np.arange(BIN_RANGE[0], BIN_RANGE[1], 1)
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        
        for inputs, label_vect, true_age, _ in train_loader:
            inputs, label_vect = inputs.to(device), label_vect.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)[0].reshape(inputs.size(0), -1)
            
            loss = criterion(outputs, label_vect)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        fold_val_preds = []
        fold_val_trues = []
        
        with torch.no_grad():
            for inputs, label_vect, true_age, _ in val_loader:
                inputs, label_vect = inputs.to(device), label_vect.to(device)
                outputs = model(inputs)[0].reshape(inputs.size(0), -1)
                
                loss = criterion(outputs, label_vect)
                val_loss += loss.item() * inputs.size(0)
                
                probs = torch.exp(outputs).cpu().numpy()
                preds = np.dot(probs, bin_centers)
                
                fold_val_preds.extend(preds)
                fold_val_trues.extend(true_age.numpy())
                
        val_loss /= len(val_loader.dataset)
        val_mae = np.mean(np.abs(np.array(fold_val_trues) - np.array(fold_val_preds)))
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MAE: {val_mae:.2f}")
        
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            epochs_no_improve = 0
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), best_model_path)
            else:
                torch.save(model.state_dict(), best_model_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                break
                
    print(f"Best Val MAE for {val_site}: {best_val_mae:.2f} years")
    
    # Ricarica e valuta il miglior modello per estrarre le predizioni definitive
    best_model = SFCN(output_dim=OUTPUT_DIM)
    best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    if torch.cuda.device_count() > 1:
        best_model = nn.DataParallel(best_model)
    best_model = best_model.to(device)
    best_model.eval()
    
    final_preds = []
    final_trues = []
    with torch.no_grad():
        for inputs, label_vect, true_age, _ in val_loader:
            inputs = inputs.to(device)
            outputs = best_model(inputs)[0].reshape(inputs.size(0), -1)
            probs = torch.exp(outputs).cpu().numpy()
            final_preds.extend(np.dot(probs, bin_centers))
            final_trues.extend(true_age.numpy())
            
    return best_val_mae, np.array(final_trues), np.array(final_preds), val_site

# --- ESECUZIONE DELLA CROSS VALIDATION ---
site_results = []
all_trues = []
all_preds = []
all_site_labels = []

for val_site in ALL_SITES:
    train_sites = [s for s in ALL_SITES if s != val_site]
    mae, t_trues, t_preds, site_name = train_site_model(train_sites, val_site)
    if mae is not None:
        site_results.append((val_site, mae))
        all_trues.extend(t_trues)
        all_preds.extend(t_preds)

        all_site_labels.extend([site_name]*len(t_trues))

## 4. Risultati Globali di Validazione (Senza Bias Correction)

In [ ]:
if len(site_results) > 0:
    print("\n==============================================")
    print(" LEAVE-ONE-SITE-OUT CROSS VALIDATION SUMMARY")
    print("==============================================")
    for site, mae in site_results:
        print(f"Site {site:4s} -> Validation MAE: {mae:.2f} years")
        
    global_mae = np.mean(np.abs(np.array(all_trues) - np.array(all_preds)))
    global_corr, _ = pearsonr(all_trues, all_preds)
    print(f"\nGLOBAL (Mean across all sites) -> MAE: {global_mae:.2f} years | Pearson r: {global_corr:.3f}")
    
    # --- PLOT RISULTATI ---
    plt.figure(figsize=(10, 8))
    
    colors = {'Guys': 'blue', 'HH': 'green', 'IOP': 'red'}
    
    for site in ALL_SITES:
        # Filtriamo i pazienti di questo sito
        site_trues = [all_trues[i] for i in range(len(all_site_labels)) if all_site_labels[i] == site]
        site_preds = [all_preds[i] for i in range(len(all_site_labels)) if all_site_labels[i] == site]
        
        if len(site_trues) > 0:
            site_mae = np.mean(np.abs(np.array(site_trues) - np.array(site_preds)))
            plt.scatter(site_trues, site_preds, alpha=0.6, color=colors.get(site, 'black'), label=f"{site} (MAE: {site_mae:.2f})")
            
    # Linea ideale y=x
    min_age = min(all_trues)
    max_age = max(all_trues)
    plt.plot([min_age, max_age], [min_age, max_age], 'k--', lw=2, label='Ideal Prediction (y=x)')
    
    plt.title(f"Leave-One-Site-Out Validation (No Bias Correction)\nGlobal MAE: {global_mae:.2f} | Global r: {global_corr:.3f}", fontsize=14, pad=15)
    plt.xlabel("True Age (Years)", fontsize=12)
    plt.ylabel("Predicted Age (Years)", fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'site_cross_validation_results.png'), dpi=300)
    plt.show()
    
    # --- SALVATAGGIO IN CSV ---
    df_results = pd.DataFrame({
        'Site': all_site_labels,
        'True_Age': all_trues,
        'Predicted_Age': all_preds,
        'Absolute_Error': np.abs(np.array(all_trues) - np.array(all_preds))
    })
    
    csv_out_path = os.path.join(PLOTS_DIR, 'ixi_cross_validation_predictions.csv')
    df_results.to_csv(csv_out_path, index=False)
    print(f"\nFull results saved to: {csv_out_path}")
else:

    print("No results to plot.")